In [ ]:
# Parameters (Papermill will override this)
RESULT_FOLDER_NAME = None

In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg
from configs.paths import DATA_DIR, PROCESSED_DIR, RESULT_DIR

import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
import pickle
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import geopandas as gpd
import osmnx as ox
import matplotlib
import matplotlib.colors as mcolors
from datetime import datetime
from collections import defaultdict
import types
from PIL import Image
import io
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from collections import defaultdict
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection

Load result and other data

In [ ]:
# fallback to latest result for manual run
# or set RESULT_FOLDER_NAME to a desired folder
if RESULT_FOLDER_NAME is None:
    result_dir = DATA_DIR / "result"
    RESULT_FOLDER_NAME = sorted([p.name for p in result_dir.iterdir() if p.is_dir()])[-1]

folder_path = RESULT_DIR / RESULT_FOLDER_NAME

with open(folder_path / "flow_result.pkl", "rb") as f:
    flow_result = pickle.load(f)

In [ ]:
def build_node_lookups(nodes):
    mode_lookup = nodes.groupby("osmid")["transport_mode"].first()
    pop_lookup = nodes.groupby("osmid")["pop_total"].sum().fillna(0)
    capacity_lookup = nodes.groupby("osmid")["shelter_capacity"].max().fillna(1e6)
    vehicle_occ_lookup = nodes.groupby("osmid")["vehicle_occ"].first().fillna(1)

    return {
        "transport_mode": mode_lookup,
        "population": pop_lookup,
        "shelter_capacity": capacity_lookup,
        "vehicle_occ": vehicle_occ_lookup,
    }

In [ ]:
nodes_df = pd.read_parquet(folder_path / "nodes.parquet")
edges_df = pd.read_parquet(folder_path / "edges.parquet")

lookups = build_node_lookups(nodes_df)

timeline_df = pd.read_parquet(folder_path / "flood_simulation_timeline.parquet")
time_names = 't' + timeline_df['step_id'].astype(str)
time_stamps = timeline_df["timestamp"]
timestep_hour = [
    (time_stamps[i + 1] - time_stamps[i]).total_seconds() / 3600.0
    for i in range(len(time_stamps) - 1)
]

## Shelters and Exits

In [ ]:
# Shelters and Exits location and capacity

_thai_fonts = [f.name for f in fm.fontManager.ttflist if "tahoma" in f.name.lower()]
if _thai_fonts:
    plt.rcParams["font.family"] = _thai_fonts[0]
else:
    for _fn in ["Browallia New", "Angsana New", "Cordia New", "Leelawadee UI"]:
        if any(_fn.lower() in f.name.lower() for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = _fn
            break


nodes_gdf = gpd.GeoDataFrame(
    nodes_df,
    geometry=gpd.points_from_xy(nodes_df["x"], nodes_df["y"]),
    crs="EPSG:4326"
)

shelters = nodes_gdf[nodes_gdf.get("is_shelter", False)]\
    .rename(columns={"shelter_names": "name"})
exits = nodes_gdf[
    nodes_gdf["is_exit"]
    if "is_exit" in nodes_gdf
    else pd.Series(False, index=nodes_gdf.index)
].rename(columns={"exit_names": "name"})

fig, ax = plt.subplots(figsize=(10, 10))
G = ox.load_graphml(PROCESSED_DIR / "hatyai_graph_with_dest.graphml")
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    node_size=0,
    edge_color="#999999",
    edge_linewidth=0.5,
    bgcolor="white"
)

ax.set_facecolor("white")

shelters.plot(ax=ax, color="blue", markersize=30, label="Shelter")
if not exits.empty:
    exits.plot(ax=ax, color="orange", markersize=30, label="Exit")

def add_labels(gdf, ax, is_shelter=False):
    for _, row in gdf.iterrows():
        name = row.get("name")
        if not name: continue

        x, y = row.geometry.x, row.geometry.y
        if is_shelter:
            cap = int(row.get("shelter_capacity", "N/A"))
            label = f"{name}\nCapacity: {cap}"
        else:
            label = name

        ax.annotate(
            label,
            xy=(x, y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=9,
            ha="left",
            va="bottom",
            bbox=dict(
                boxstyle="round,pad=0.3",
                fc="white",
                alpha=0.9,
                lw=0.5,
                ec="gray",
            ),
        )

add_labels(shelters, ax, True)
add_labels(exits, ax)

plt.tight_layout()
plt.legend(loc="lower left")
plt.title("Shelters and exits Map")
ax.set_axis_off()
plt.savefig(folder_path / "shelters_and_exits.png", dpi=200, bbox_inches="tight", pad_inches=0.1)
plt.show()

## Flood Depth Visualization

In [ ]:
# ── 1. Time-series of aggregate stats ──────────────────────────────────────

depth_cols = [c for c in edges_df.columns if c.startswith("flood_depth_t")]
depth_matrix = edges_df[depth_cols].values  # (n_edges, n_timesteps)


mean_depth = depth_matrix.mean(axis=0)
max_depth  = depth_matrix.max(axis=0)
frac_flooded = (depth_matrix > 0).mean(axis=0)  # fraction of edges with any flood

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(time_stamps, mean_depth, color="steelblue")
axes[0].set_ylabel("Mean depth (m)")
axes[0].set_title("Average flood depth across all edges")

axes[1].plot(time_stamps, max_depth, color="crimson")
axes[1].set_ylabel("Max depth (m)")
axes[1].set_title("Maximum flood depth across all edges")

axes[2].plot(time_stamps, frac_flooded * 100, color="darkorange")
axes[2].set_ylabel("% edges flooded")
axes[2].set_title("Fraction of edges with flood depth > 0")
axes[2].set_xlabel("Time")

for ax in axes:
    for ts in time_stamps:
        ax.axvline(ts, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig(folder_path / "flood_timeseries.png", dpi=150)
plt.show()

In [ ]:
# ── 2. Spatial snapshots at key timesteps ──────────────────────────────────
# Pick a few representative steps: start, ~25%, ~50%, ~75%, end
n_steps = depth_matrix.shape[1]
snap_indices = [0, n_steps // 4, n_steps // 2, 3 * n_steps // 4, n_steps - 1]

edges_geo = gpd.GeoDataFrame(
    edges_df, 
    geometry=gpd.GeoSeries.from_wkt(edges_df["geometry"]),
    crs="EPSG:4326"
)

vmax = depth_matrix.max()
norm = mcolors.Normalize(vmin=0, vmax=vmax if vmax > 0 else 1)
cmap = matplotlib.colormaps["YlOrRd"]

fig, axes = plt.subplots(1, len(snap_indices), figsize=(20, 5))

for ax, ti in zip(axes, snap_indices):
    col = f"flood_depth_t{ti}"
    depths = edges_geo[col] if col in edges_geo.columns else pd.Series(0.0, index=edges_geo.index)

    # Dry edges in light gray, flooded edges colored by depth
    dry = edges_geo[depths <= 0]
    wet = edges_geo[depths > 0]

    dry.plot(ax=ax, color="lightgray", linewidth=0.4, aspect=None)
    if not wet.empty:
        wet.plot(ax=ax, column=col, cmap=cmap, norm=norm, linewidth=1.2, aspect=None)

    ax.set_aspect("equal")
    ts_label = time_stamps[ti].strftime("%Y-%m-%d %H:%M")
    ax.set_title(ts_label, fontsize=9)
    ax.set_axis_off()

sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, orientation="vertical", fraction=0.015, pad=0.02, label="Flood depth (m)")
plt.suptitle("Flood depth spatial distribution", y=1.02)
plt.savefig(folder_path / "flood_spatial_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()

## Evacuation Visualization

In [ ]:
# ── Time info ────────────────────────────────────────────────────
n_timesteps = len(time_names)
sim_start_dt = pd.Timestamp(time_stamps.iloc[0])
sim_end_dt   = pd.Timestamp(time_stamps.iloc[-1])
sim_duration_h = (sim_end_dt - sim_start_dt).total_seconds() / 3600.0

# ── Unpack flow_result ───────────────────────────────────────────────────
total_evacuated      = flow_result["flow_value"]
evac_rate            = flow_result["evac_rate"]
shelter_flow_people  = flow_result["shelter_flow_people"]   # {osmid: float}
exit_flow_people     = flow_result["exit_flow_people"]       # {osmid: float}
flow_dict            = flow_result["flow_dict"]
evac_completion_ts   = flow_result.get("evac_completion_timestamp")
evac_completion_h    = flow_result.get("evac_completion_hours")
evac_completion_layer= flow_result.get("evac_completion_layer")


flood_cols  = [f'flood_depth_t{tid}' for tid in timeline_df['step_id'].iloc[:-1]]

### Reports

In [ ]:
def compute_bottleneck_nodes(flow_dict, nodes, top_n=10):
    """
    Bottleneck nodes: non-destination nodes with highest total
    throughput flow (sum of all outgoing move-edge flows).
    """
    shelter_osmids = set(nodes.loc[nodes["is_shelter"] == True, "osmid"].astype(int))
    exit_osmids = (
        set(nodes.loc[nodes["is_exit"] == True, "osmid"].astype(int))
        if "is_exit" in nodes.columns else set()
    )
    destination_osmids = shelter_osmids | exit_osmids

    # Sum outgoing flow per osmid across all time layers
    node_throughput = defaultdict(float)
    for u, nbrs in flow_dict.items():
        if not isinstance(u, tuple):
            continue
        osmid_u, _ = u
        if osmid_u in destination_osmids:
            continue
        for v, f in nbrs.items():
            if f > 0 and isinstance(v, tuple):  # move or wait edge to another layer
                osmid_v, _ = v
                if osmid_v != osmid_u:           # exclude wait edges (same osmid)
                    node_throughput[osmid_u] += f

    df = pd.DataFrame([
        {"osmid": osmid, "total_flow": flow}
        for osmid, flow in node_throughput.items()
    ]).sort_values("total_flow", ascending=False).head(top_n).reset_index(drop=True)

    # Join node metadata
    node_meta = nodes.drop_duplicates("osmid").set_index("osmid")[
        ["shelter_names", "is_shelter"]
        + (["exit_names", "is_exit"] if "is_exit" in nodes.columns else [])
    ]
    df = df.join(node_meta, on="osmid", how="left")
    df.index += 1  # rank from 1
    df.index.name = "rank"
    return df


def compute_bottleneck_edges(flow_dict, edges, top_n=10):
    """
    Bottleneck edges: physical road edges (u, v osmid pair) ranked by:
      - total_flow  : cumulative flow across all timesteps
      - max_sat     : highest single-timestep flow/capacity ratio
      - mean_sat    : average saturation across timesteps where flow > 0
    """
    # Aggregate flow per (u_osmid, v_osmid) across all time layers
    edge_flow   = defaultdict(float)
    edge_cap    = {}      # store capacity per (u, v) — same across layers
    edge_sat    = defaultdict(list)  # saturation samples per (u, v)

    for u, nbrs in flow_dict.items():
        if not isinstance(u, tuple):
            continue
        osmid_u, _ = u
        for v, f in nbrs.items():
            if not isinstance(v, tuple):
                continue
            osmid_v, _ = v
            if osmid_v == osmid_u:   # skip wait edges
                continue
            if f > 0:
                key = (osmid_u, osmid_v)
                edge_flow[key] += f

                # Get capacity from the graph edge attributes
                cap = flow_dict.get(u, {})  # we'll get cap from edges df instead
                edge_cap[key] = None  # resolved below

    # Resolve capacity from edges DataFrame
    cap_lookup = (
        edges[["u", "v", "capacity"]]
        .dropna(subset=["capacity"])
        .drop_duplicates(["u", "v"])
        .set_index(["u", "v"])["capacity"]
        .to_dict()
    )

    rows = []
    for (u, v), total_flow in edge_flow.items():
        cap = cap_lookup.get((u, v)) or cap_lookup.get((v, u))
        sat = (total_flow / cap) if cap and cap > 0 else None
        rows.append({
            "u": u,
            "v": v,
            "total_flow": total_flow,
            "capacity": cap,
            "total_sat": sat,   # cumulative flow / capacity (volume pressure)
        })

    df = pd.DataFrame(rows)

    # max single-layer saturation
    layer_sat = defaultdict(list)
    for u_node, nbrs in flow_dict.items():
        if not isinstance(u_node, tuple):
            continue
        osmid_u, ti = u_node
        for v_node, f in nbrs.items():
            if not isinstance(v_node, tuple):
                continue
            osmid_v, _ = v_node
            if osmid_v == osmid_u:
                continue
            if f > 0:
                key = (osmid_u, osmid_v)
                cap = cap_lookup.get(key) or cap_lookup.get((osmid_v, osmid_u))
                if cap and cap > 0:
                    layer_sat[key].append(f / cap)

    df["max_sat"]  = df.apply(
        lambda r: max(layer_sat.get((r.u, r.v), [0])), axis=1
    )
    df["mean_sat"] = df.apply(
        lambda r: np.mean(layer_sat.get((r.u, r.v), [0])), axis=1
    )

    # Rank by total_flow, show both metrics
    df = df.sort_values("total_flow", ascending=False).head(top_n).reset_index(drop=True)
    df.index += 1
    df.index.name = "rank"

    # Format saturation as percentage strings for display
    df["total_sat%"] = df["total_sat"].apply(
        lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—"
    )
    df["max_sat%"]  = df["max_sat"].apply(lambda x: f"{x*100:.1f}%")
    df["mean_sat%"] = df["mean_sat"].apply(lambda x: f"{x*100:.1f}%")

    return df


# ── Run ───────────────────────────────────────────────────────────────────────
TOP_N = 10

bottleneck_nodes_df = compute_bottleneck_nodes(flow_dict, nodes_df, top_n=TOP_N)
bottleneck_edges_df = compute_bottleneck_edges(flow_dict, edges_df, top_n=TOP_N)

bottleneck_nodes_df.to_parquet(folder_path / "bottleneck_nodes.parquet")
bottleneck_edges_df.to_parquet(folder_path / "bottleneck_edges.parquet")
print(f"Saved bottleneck_nodes.parquet and bottleneck_edges.parquet to {folder_path}")

Creating report

In [ ]:
# ── Population totals ────────────────────────────────────────────────────
total_population  = float(nodes_df["pop_total"].fillna(0).sum())
total_stranded    = total_population - total_evacuated
evac_shelter_total = sum(shelter_flow_people.values())
evac_exit_total    = sum(exit_flow_people.values())

# ── Isolated nodes (all movement edges flooded at every timestep) ────────
flood_cols = [c for c in edges_df.columns if c.startswith("flood_depth_t")]
impassable_depth = cfg.IMPASSABLE_FLOOD_DEPTH  # adjust if cfg not in scope

# A node is isolated if every edge incident to it is impassable
# at every timestep — check u side (bidirectional edges handled by data)
edge_ever_passable = (
    edges_df[flood_cols].lt(impassable_depth).any(axis=1)
)
passable_edges = edges_df[edge_ever_passable]
passable_nodes = set(passable_edges["u"]).union(set(passable_edges["v"]))

nodes_with_pop = nodes_df[nodes_df["pop_total"].fillna(0) > 0]
isolated_nodes = nodes_with_pop[
    ~nodes_with_pop["osmid"].isin(passable_nodes)
]
isolated_population = float(isolated_nodes["pop_total"].fillna(0).sum())
isolated_count = len(isolated_nodes)

# ── Evacuation start time (first nonzero flow layer) ─────────────────────
first_flow_layer = None
for u, nbrs in flow_dict.items():
    if not isinstance(u, tuple):
        continue
    osmid_u, layer_u = u
    for v, f in nbrs.items():
        if f > 0 and isinstance(v, str):  # reached a collector
            if first_flow_layer is None or layer_u < first_flow_layer:
                first_flow_layer = layer_u
evac_start_ts = (
    pd.Timestamp(time_stamps.iloc[first_flow_layer])
    if first_flow_layer is not None else None
)

# ── Per-timestep flooded edge count ──────────────────────────────────────
flooded_per_step = {}
for i, col in enumerate(flood_cols):
    flooded_per_step[i] = int((edges_df[col].fillna(0) >= impassable_depth).sum())

peak_flood_layer = max(flooded_per_step, key=flooded_per_step.get)
peak_flood_count = flooded_per_step[peak_flood_layer]
peak_flood_ts    = pd.Timestamp(time_stamps.iloc[peak_flood_layer])

# ── Shelter details ───────────────────────────────────────────────────────
shelter_nodes = nodes_df[nodes_df["is_shelter"] == True].drop_duplicates("osmid")
shelter_rows = []
for _, row in shelter_nodes.iterrows():
    osmid    = int(row["osmid"])
    name     = row.get("shelter_names", f"Shelter {osmid}")
    capacity = float(row["shelter_capacity"]) if pd.notna(row["shelter_capacity"]) else None
    received = shelter_flow_people.get(osmid, 0.0)
    util_pct = (received / capacity * 100) if capacity and capacity > 0 else None
    over_cap = capacity is not None and received > capacity
    shelter_rows.append({
        "osmid": osmid, "name": name, "capacity": capacity,
        "received": received, "util_pct": util_pct, "over_cap": over_cap,
    })

# ── Exit details ──────────────────────────────────────────────────────────
exit_nodes = nodes_df[nodes_df.get("is_exit", pd.Series(False, index=nodes_df.index)) == True]
if "is_exit" in nodes_df.columns:
    exit_nodes = nodes_df[nodes_df["is_exit"] == True].drop_duplicates("osmid")
else:
    exit_nodes = pd.DataFrame()

exit_rows = []
for _, row in exit_nodes.iterrows():
    osmid    = int(row["osmid"])
    name     = row.get("exit_names", f"Exit {osmid}")
    received = exit_flow_people.get(osmid, 0.0)
    exit_rows.append({"osmid": osmid, "name": name, "received": received})

# ── Cumulative evacuation milestones ─────────────────────────────────────
# Count people arriving at collectors per layer
layer_flow = np.zeros(n_timesteps)
for u, nbrs in flow_dict.items():
    if not isinstance(u, tuple):
        continue
    _, layer_u = u
    for v, f in nbrs.items():
        if f > 0 and isinstance(v, str):
            if layer_u < n_timesteps:
                layer_flow[layer_u] += f

cumulative = np.cumsum(layer_flow)
milestones = {}
for pct in [25, 50, 75, 100]:
    target = (pct / 100.0) * total_evacuated
    layers_hit = np.where(cumulative >= target)[0]
    if len(layers_hit) > 0:
        li = layers_hit[0]
        milestones[pct] = pd.Timestamp(time_stamps.iloc[li])

# ── Waiting Edges Congestion ──────────────────────────────────────────────
node_time_volume = defaultdict(float)
for u in flow_dict:
    for v, flow in flow_dict[u].items():
        if flow <= 0:
            continue

        # check if it's a waiting edge
        if isinstance(u, tuple) and isinstance(v, tuple):
            osmid_u, t_u = u
            osmid_v, t_v = v

            if osmid_u == osmid_v and t_v == t_u + 1:
                # this is a waiting edge
                node_time_volume[(osmid_u, t_u)] += flow

top_nodes = sorted(
    node_time_volume.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

# ── Warnings ──────────────────────────────────────────────────────────────
warnings = []
for s in shelter_rows:
    if s["over_cap"]:
        warnings.append(f"⚠ Shelter '{s['name']}' exceeded capacity "
                        f"({s['received']:.0f} / {s['capacity']:.0f})")
if total_stranded > 0.01 * total_population:
    warnings.append(f"⚠ {total_stranded:.0f} people ({total_stranded/total_population:.1%}) "
                    f"did not evacuate")
if isolated_count > 0:
    warnings.append(f"⚠ {isolated_count} node(s) with {isolated_population:.0f} people "
                    f"were fully isolated by flooding")

# ═════════════════════════════════════════════════════════════════════════
# Build report string
# ═════════════════════════════════════════════════════════════════════════
W = 62  # report width
DIV  = "─" * W
DDIV = "═" * W

def fmt_ts(ts):
    return pd.Timestamp(ts).strftime("%Y-%m-%d %H:%M") if ts is not None else "N/A"

def section(title):
    return f"\n{DIV}\n  {title}\n{DIV}"

lines = []
lines.append(DDIV)
lines.append("  EVACUATION SIMULATION REPORT")
lines.append(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"  Result    : {folder_path.name}")
lines.append(DDIV)

# 1. Simulation overview
lines.append(section("1. SIMULATION OVERVIEW"))
lines.append(f"  Timesteps            : {n_timesteps}")
lines.append(f"  Simulation start     : {fmt_ts(sim_start_dt)}")
lines.append(f"  Simulation end       : {fmt_ts(sim_end_dt)}")
lines.append(f"  Simulation duration  : {sim_duration_h:.1f} h")
lines.append(f"  Impassable depth     : {impassable_depth:.2f} m")

# 2. Evacuation outcomes
lines.append(section("2. EVACUATION OUTCOMES"))
lines.append(f"  Total population     : {total_population:>10,.0f}")
lines.append(f"  Total evacuated      : {total_evacuated:>10,.0f}  ({evac_rate:.1%})")
lines.append(f"    → via shelters     : {evac_shelter_total:>10,.0f}")
lines.append(f"    → via exits        : {evac_exit_total:>10,.0f}")
lines.append(f"  Stranded (total)     : {total_stranded:>10,.0f}  ({total_stranded/total_population:.1%})")
lines.append(f"    → isolated nodes   : {isolated_count} node(s), {isolated_population:,.0f} people")

# 3. Temporal analysis
lines.append(section("3. TEMPORAL ANALYSIS"))
lines.append(f"  Evacuation start     : {fmt_ts(evac_start_ts)}"
                + (f"  [t{first_flow_layer}]" if first_flow_layer is not None else ""))
lines.append(f"  Evacuation end       : {fmt_ts(evac_completion_ts)}"
                + (f"  [t{evac_completion_layer}]" if evac_completion_layer is not None else ""))
lines.append(f"  Evacuation duration  : "
                + (f"{evac_completion_h:.2f} h" if evac_completion_h is not None else "N/A"))
lines.append(f"\n  Cumulative milestones:")
for pct, ts in milestones.items():
    lines.append(f"    {pct:>3}% evacuated at : {fmt_ts(ts)}")

# 4. Shelter capacity report
lines.append(section("4. SHELTER CAPACITY REPORT"))
col_w = [max(len(str(s["name"])) for s in shelter_rows) + 2 if shelter_rows else 20, 10, 10, 8, 6]
col_w[0] = max(col_w[0], 20)
hdr = (f"  {'Name':<{col_w[0]}} {'Capacity':>{col_w[1]}} "
        f"{'Received':>{col_w[2]}} {'Util%':>{col_w[3]}} {'Flag':>{col_w[4]}}")
lines.append(hdr)
lines.append("  " + "·" * (sum(col_w) + 5))
for s in sorted(shelter_rows, key=lambda x: -x["received"]):
    cap_str  = f"{s['capacity']:,.0f}" if s["capacity"] is not None else "unlimited"
    util_str = f"{s['util_pct']:.1f}%" if s["util_pct"] is not None else "—"
    flag     = "OVER" if s["over_cap"] else ""
    lines.append(
        f"  {str(s['name']):<{col_w[0]}} {cap_str:>{col_w[1]}} "
        f"{s['received']:>{col_w[2]},.0f} {util_str:>{col_w[3]}} {flag:>{col_w[4]}}"
    )
total_shelter_cap = sum(s["capacity"] for s in shelter_rows if s["capacity"] is not None)
lines.append("  " + "·" * (sum(col_w) + 5))
lines.append(f"  {'TOTAL':<{col_w[0]}} {total_shelter_cap:>{col_w[1]},.0f} "
                f"{evac_shelter_total:>{col_w[2]},.0f}")

# 5. Exit report
if exit_rows:
    lines.append(section("5. EXIT REPORT"))
    ex_col_w = max((len(str(e["name"])) for e in exit_rows), default=20) + 2
    ex_col_w = max(ex_col_w, 20)
    lines.append(f"  {'Name':<{ex_col_w}} {'Routed':>10}")
    lines.append("  " + "·" * (ex_col_w + 12))
    for e in sorted(exit_rows, key=lambda x: -x["received"]):
        lines.append(f"  {str(e['name']):<{ex_col_w}} {e['received']:>10,.0f}")
    lines.append("  " + "·" * (ex_col_w + 12))
    lines.append(f"  {'TOTAL':<{ex_col_w}} {evac_exit_total:>10,.0f}")

# 6. Network stress
sec_n = 6 if exit_rows else 5
lines.append(section(f"{sec_n}. NETWORK STRESS"))
sec_n += 1
lines.append(f"  Total edges          : {len(edges_df)}")
lines.append(f"  Peak flooded edges   : {peak_flood_count} at {fmt_ts(peak_flood_ts)}  [t{peak_flood_layer}]")
lines.append(f"  Fully isolated nodes : {isolated_count} ({isolated_population:,.0f} people)")

# ── Bottleneck section ────────────────────────────────────────────────────
lines.append(section(f"{sec_n}. TOP BOTTLENECK NODES"))
sec_n += 1
bn_node_cols = ["osmid", "total_flow"]
hdr = f"  {'Rank':>4}  {'osmid':>12}  {'Total Flow':>12}"
lines.append(hdr)
lines.append("  " + "·" * 34)
for rank, row in bottleneck_nodes_df.iterrows():
    lines.append(
        f"  {rank:>4}  {int(row['osmid']):>12}  {row['total_flow']:>12,.0f}"
    )

lines.append(section(f"{sec_n}. TOP BOTTLENECK EDGES"))
sec_n += 1
hdr = (f"  {'Rank':>4}  {'u':>12}  {'v':>12}  "
       f"{'Total Flow':>12}  {'Cap':>10}  "
       f"{'TotalSat%':>10}  {'MaxSat%':>8}  {'MeanSat%':>9}")
lines.append(hdr)
lines.append("  " + "·" * 88)
for rank, row in bottleneck_edges_df.iterrows():
    lines.append(
        f"  {rank:>4}  {int(row['u']):>12}  {int(row['v']):>12}  "
        f"{row['total_flow']:>12,.0f}  "
        f"{row['capacity']:>10,.0f}  "
        f"{row['total_sat%']:>10}  {row['max_sat%']:>8}  {row['mean_sat%']:>9}"
    )

# ── Waiting Edges Congestion ──────────────────────────────────────────────
lines.append(section(f"{sec_n}. WAITING EDGES CONGESTION"))
sec_n += 1
for (osmid, t), vol in top_nodes:
    lines.append(f"Node {osmid} at time {t}: {vol}")

# 7. Warnings
lines.append(section(f"{sec_n}. WARNINGS & FLAGS"))
if warnings:
    for w in warnings:
        lines.append(f"  {w}")
else:
    lines.append("  ✓ No warnings.")

lines.append(f"\n{DDIV}\n")

report_str = "\n".join(lines)

# ── Save ─────────────────────────────────────────────────────────────────
out_path = folder_path / "evacuation_report.md"
out_path.write_text(report_str, encoding="utf-8")
print(f"Report saved to: {out_path}")

### Bottleneck Visualization

In [ ]:
def plot_bottlenecks(
    nodes, edges,
    bottleneck_nodes_df,
    bottleneck_edges_df,
    result_folder,
    top_n_nodes=10,
    top_n_edges=10,
    figsize=(14, 12),
):
    """
    Plot top bottleneck nodes and edges on a static map.

    - Edges: drawn with width + color intensity scaled to total_flow
             max_sat shown via edge alpha
    - Nodes: drawn as circles sized by total_flow, ranked and labelled
    - Grey base network drawn underneath for context
    """
    fig, ax = plt.subplots(figsize=figsize, facecolor="white")

    # ── Node coordinate lookup ────────────────────────────────────────────
    node_xy = (
        nodes.drop_duplicates("osmid")
        .set_index("osmid")[["x", "y"]]
        .to_dict("index")
    )

    # ── 1. Draw grey base network ─────────────────────────────────────────
    for _, row in edges.iterrows():
        u_xy = node_xy.get(int(row["u"]))
        v_xy = node_xy.get(int(row["v"]))
        if u_xy and v_xy:
            ax.plot(
                [u_xy["x"], v_xy["x"]],
                [u_xy["y"], v_xy["y"]],
                color="#cccccc", linewidth=0.4, zorder=1,
            )

    # ── 2. Draw bottleneck edges ──────────────────────────────────────────
    bn_edges = bottleneck_edges_df.head(top_n_edges)
    max_edge_flow = bn_edges["total_flow"].max()

    edge_cmap   = cm.get_cmap("YlOrRd")
    edge_norm   = mcolors.Normalize(
        vmin=bn_edges["total_flow"].min(),
        vmax=max_edge_flow,
    )

    for rank, row in bn_edges.iterrows():
        u_xy = node_xy.get(int(row["u"]))
        v_xy = node_xy.get(int(row["v"]))
        if not (u_xy and v_xy):
            continue

        flow_norm  = edge_norm(row["total_flow"])
        color      = edge_cmap(flow_norm)
        linewidth  = 4 + 5.0 * flow_norm          # 1.5 – 6.5
        alpha      = 0.5 + 0.5 * row["max_sat"]     # brighter = more saturated

        ax.plot(
            [u_xy["x"], v_xy["x"]],
            [u_xy["y"], v_xy["y"]],
            color=color, linewidth=linewidth,
            alpha=min(alpha, 1.0), zorder=3,
            solid_capstyle="round",
        )
        # rank label at midpoint
        mx = (u_xy["x"] + v_xy["x"]) / 2
        my = (u_xy["y"] + v_xy["y"]) / 2
        ax.text(
            mx, my, f"E{rank}",
            fontsize=8, color="saddlebrown",
            ha="center", va="center", zorder=5,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.6),
        )

    # ── 3. Draw bottleneck nodes ──────────────────────────────────────────
    bn_nodes = bottleneck_nodes_df.head(top_n_nodes)
    max_node_flow = bn_nodes["total_flow"].max()

    node_cmap = cm.get_cmap("Blues")
    node_norm = mcolors.Normalize(
        vmin=bn_nodes["total_flow"].min(),
        vmax=max_node_flow,
    )

    for rank, row in bn_nodes.iterrows():
        xy = node_xy.get(int(row["osmid"]))
        if not xy:
            continue

        flow_norm  = node_norm(row["total_flow"])
        color      = node_cmap(0.4 + 0.6 * flow_norm)   # avoid too-light blues
        markersize = 100 + 180 * flow_norm                 # 40 – 200 pt²

        ax.scatter(
            xy["x"], xy["y"],
            s=markersize, color=color,
            edgecolors="navy", linewidths=0.8,
            zorder=4,
        )
        ax.text(
            xy["x"], xy["y"], f"N{rank}",
            fontsize=8, color="white", fontweight="bold",
            ha="center", va="center", zorder=6,
        )

    # ── 4. Colorbars ──────────────────────────────────────────────────────
    sm_edge = cm.ScalarMappable(cmap=edge_cmap, norm=edge_norm)
    sm_edge.set_array([])
    cbar_e = fig.colorbar(sm_edge, ax=ax, fraction=0.02, pad=0.01, shrink=0.45,
                          location="right")
    cbar_e.set_label("Edge cumulative flow (people)", fontsize=8)
    cbar_e.ax.tick_params(labelsize=7)

    sm_node = cm.ScalarMappable(cmap=node_cmap,
                                 norm=mcolors.Normalize(
                                     vmin=bn_nodes["total_flow"].min(),
                                     vmax=max_node_flow))
    sm_node.set_array([])
    cbar_n = fig.colorbar(sm_node, ax=ax, fraction=0.02, pad=0.06, shrink=0.45,
                          location="right")
    cbar_n.set_label("Node throughput flow (people)", fontsize=8)
    cbar_n.ax.tick_params(labelsize=7)

    # ── 5. Legend & labels ────────────────────────────────────────────────
    legend_handles = [
        Line2D([0], [0], color="#aaaaaa", linewidth=1,   label="Road network"),
        Line2D([0], [0], color="orangered", linewidth=3, label="Bottleneck edge"),
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor="steelblue", markersize=9,
               markeredgecolor="navy",      label="Bottleneck node"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=8,
              framealpha=0.9, edgecolor="#cccccc")

    ax.set_title(
        f"Top-{top_n_nodes} Bottleneck Nodes  &  Top-{top_n_edges} Bottleneck Edges\n"
        f"Edge color/width = cumulative flow  |  Edge alpha = max saturation",
        fontsize=10, pad=10,
    )
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude",  fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor("white")

    plt.tight_layout()

    # ── Save ─────────────────────────────────────────────────────────────
    out_path = result_folder / "bottleneck_map.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight", facecolor="white")
    print(f"Saved: {out_path}")
    plt.show()
    return fig, ax


# ── Run ───────────────────────────────────────────────────────────────────────
fig, ax = plot_bottlenecks(
    nodes_df, edges_df,
    bottleneck_nodes_df,
    bottleneck_edges_df,
    result_folder=folder_path,
    top_n_nodes=TOP_N,
    top_n_edges=TOP_N,
)

### Animation

In [ ]:
OUTPUT_GIF = Path("evacuation_animation.gif")
VIZ = types.SimpleNamespace(
    FIG_W              = 12,      # figure width  (inches)
    FIG_H          = 10,      # figure height (inches)
    DPI            = 120,     # render DPI (lower = smaller file / faster)
    FRAME_DURATION = 100,     # ms per frame in GIF
    # nodes
    BASE_NODE_SIZE  = 6,   # scatter marker area at max inflow
    AGENT_SIZE      = 20,
    # edges
    EDGE_ALPHA     = 0.9,    # normal edge transparency
    CLOSED_ALPHA   = 0.8,    # closed (red) edge transparency
    EDGE_LW        = 1,     # normal edge line width
    CLOSED_LW      = 1.2,     # closed edge line width
    # shelter / exit markers
    SYMBOL_SIZE    = 100,     # shelter / exit marker size
)

In [ ]:
# Spatial lookups & shelter/exit masks
nodes_u = nodes_df.drop_duplicates('osmid').set_index('osmid')

pos      = {int(osmid): (row['x'], row['y']) for osmid, row in nodes_u.iterrows()}
node_ids = np.array(list(pos.keys()))
xs       = np.array([pos[n][0] for n in node_ids])
ys       = np.array([pos[n][1] for n in node_ids])

is_shelter = {int(oid): bool(r['is_shelter']) for oid, r in nodes_u.iterrows()}
if 'is_exit' in nodes_u.columns:
    is_exit = {int(oid): bool(r['is_exit']) for oid, r in nodes_u.iterrows()}
else:
    is_exit = {}
    print("'is_exit' column not found – exits disabled.")

shelters_ids = set(int(x) for x in nodes_u[nodes_u['is_shelter']].index)
exits_ids    = (
    set(int(x) for x in nodes_u[nodes_u['is_exit']].index)
    if 'is_exit' in nodes_u.columns else set()
)

sh_mask  = np.array([is_shelter.get(n, False) for n in node_ids])
ex_mask  = np.array([is_exit.get(n, False)    for n in node_ids])
reg_mask = ~sh_mask & ~ex_mask

print(f'Shelters: {sh_mask.sum()}  |  Exits: {ex_mask.sum()}  |  Regular: {reg_mask.sum()}')

In [ ]:
# Destination colours (static across time)
def dominant_dest(osmid: int):
    """Dominant evacuation destination for a node (summed across all time layers)."""
    totals: dict = {}
    for ti in range(len(time_names)):
        for nbr, val in flow_dict.get((osmid, ti), {}).items():
            if val > 0 and isinstance(nbr, str):
                totals[nbr] = totals.get(nbr, 0) + val
    if not totals:
        return 'none'
    best = max(totals, key=totals.__getitem__)
    return int(best.split('_', 1)[1])

all_dests   = list(shelters_ids | exits_ids)
dest_cmap   = plt.cm.get_cmap('tab20', max(len(all_dests), 1))
dest_colour = {d: dest_cmap(i) for i, d in enumerate(all_dests)}
dest_colour['none'] = (0.65, 0.65, 0.65, 1.0)

print('Computing dominant destination per node…')
node_dest    = {n: dominant_dest(n) for n in node_ids}
node_colours = np.array([dest_colour.get(node_dest[n], dest_colour['none'])
                          for n in node_ids])
print('Done.')

In [ ]:
# Agent pre-computation (moving dots along real edge geometries)

# ── load graph & build edge geometry lookup ───────────────────────────────────
graph_filtered = ox.load_graphml(PROCESSED_DIR / 'hatyai_graph_clean.graphml')
edges_anim = (
    ox.graph_to_gdfs(graph_filtered, nodes=False, edges=True)
    .reset_index()[['u', 'v', 'key', 'geometry']]
)
geom_lookup = {
    (int(r.u), int(r.v)): r.geometry
    for _, r in edges_anim.iterrows()
}

# ── reverse-BFS to find every time-expanded node on an exit-bound flow path ───
H_flow          = flow_result['graph']
exit_osmids_set = set(int(x) for x in flow_result.get('exit_osmids', set()))

rev_adj_flow = defaultdict(list)
for u, nbrs in flow_dict.items():
    for v, f in nbrs.items():
        if f > 0 and isinstance(u, tuple) and isinstance(v, tuple):
            rev_adj_flow[v].append(u)

from collections import deque
exit_reachable_te = set()
bfs_q = deque()
for osmid in exit_osmids_set:
    for layer in range(len(time_names)):
        node = (int(osmid), layer)
        if node not in exit_reachable_te:
            exit_reachable_te.add(node)
            bfs_q.append(node)
while bfs_q:
    node = bfs_q.popleft()
    for prev in rev_adj_flow.get(node, []):
        if prev not in exit_reachable_te:
            exit_reachable_te.add(prev)
            bfs_q.append(prev)
print(f'Exit-bound time-expanded nodes: {len(exit_reachable_te)}')

# ── collect move edges with positive flow ─────────────────────────────────────
AGENT_SCALE = 20    # people per agent dot
AGENT_CAP   = 200   # max dots per edge

move_edges = []
for (u, v, data) in H_flow.edges(data=True):
    if data.get('kind') != 'move':
        continue
    flow_val = flow_dict.get(u, {}).get(v, 0)
    if flow_val <= 0:
        continue
    ti = u[1]
    if ti >= len(time_names):
        continue
    geom = geom_lookup.get((int(u[0]), int(v[0]))) or geom_lookup.get((int(v[0]), int(u[0])))
    if geom is None:
        continue
    travel_h   = float(data.get('travel_hours', timestep_hour[min(ti, len(timestep_hour)-1)]))
    is_exit_e  = u in exit_reachable_te
    move_edges.append((u, v, flow_val, geom, travel_h, is_exit_e))

print(f'Move edges with flow: {len(move_edges)}')

# ── interpolate agent positions for every frame each edge is active ───────────
ts_array = np.array([np.datetime64(t) for t in time_stamps])

def edge_to_agents(u, v, flow_val, geom, travel_h, is_exit_e):
    ti       = u[1]
    n_agents = int(min(max(flow_val / AGENT_SCALE, 1), AGENT_CAP))
    if travel_h <= 0:
        travel_h = 1e-6
    end_np       = np.datetime64(time_stamps[ti] + pd.to_timedelta(travel_h, unit='h'))
    arrival_frame = int(np.searchsorted(ts_array, end_np, side='left'))
    last_frame    = min(len(time_names) - 1, max(arrival_frame, ti))
    pts = []
    start_ts = time_stamps[ti]
    for frame in range(ti, last_frame + 1):
        elapsed  = (time_stamps[frame] - start_ts).total_seconds() / 3600.0
        progress = min(max(elapsed / travel_h, 0.0), 1.0)
        for j in range(n_agents):
            frac = (j + 1) / (n_agents + 1)
            pt   = geom.interpolate(frac * progress, normalized=True)
            pts.append((frame, pt.x, pt.y, is_exit_e))
    return pts

agent_chunks = Parallel(n_jobs=-1, prefer='threads')(
    delayed(edge_to_agents)(u, v, fv, geom, th, ie)
    for (u, v, fv, geom, th, ie) in move_edges
)

# ── bucket by frame and destination type ─────────────────────────────────────
# shelter-bound → blue,  exit-bound → orange  (matches original)
agent_shelter = [[] for _ in time_names]   # list of (x, y) per frame
agent_exit    = [[] for _ in time_names]
for chunk in agent_chunks:
    for frame, x, y, is_exit_e in chunk:
        if is_exit_e:
            agent_exit[frame].append((x, y))
        else:
            agent_shelter[frame].append((x, y))

n_sh = sum(len(f) for f in agent_shelter)
n_ex = sum(len(f) for f in agent_exit)
print(f'Agent dots — shelter-bound: {n_sh}  |  exit-bound: {n_ex}')
if n_sh + n_ex == 0:
    print('WARNING: no agent dots produced. Check flow_result and graph.')

In [ ]:
# Edge geometry & per-timestep flood stats
def build_edge_records(edges_df, pos, flood_cols):
    records = []
    for _, row in edges_df.iterrows():
        u, v = int(row['u']), int(row['v'])
        if u not in pos or v not in pos:
            continue
        depths = []
        for col in flood_cols:
            d = row.get(col, 0)
            depths.append(0.0 if pd.isna(d) else float(d))
        records.append({
            'x0': pos[u][0], 'y0': pos[u][1],
            'x1': pos[v][0], 'y1': pos[v][1],
            'depths': depths,
        })
    return records


def timestep_edge_stats(ti, edge_records, flood_cols, cfg):
    depth_idx = min(ti, len(flood_cols) - 1)
    open_segs, open_depths, closed_segs, all_depths = [], [], [], []
    for er in edge_records:
        d   = er['depths'][depth_idx] if depth_idx < len(er['depths']) else 0.0
        seg = (er['x0'], er['y0'], er['x1'], er['y1'])
        all_depths.append(d)
        if d >= cfg.IMPASSABLE_FLOOD_DEPTH:
            closed_segs.append(seg)
        else:
            open_segs.append(seg)
            open_depths.append(d)
    avg = float(np.mean(all_depths)) if all_depths else 0.0
    mx  = float(np.max(all_depths))  if all_depths else 0.0
    return open_segs, open_depths, closed_segs, avg, mx


edge_records   = build_edge_records(edges_df, pos, flood_cols)
all_edge_stats = [
    timestep_edge_stats(ti, edge_records, flood_cols, cfg)
    for ti in range(len(time_names))
]
global_max_depth = max(
    max((s[4] for s in all_edge_stats), default=0),
    cfg.IMPASSABLE_FLOOD_DEPTH,
)
print(f'Edge records      : {len(edge_records)}')
print(f'Global max depth  : {global_max_depth:.3f} m')

In [ ]:
# Colourmaps & legend
water_cmap = LinearSegmentedColormap.from_list(
    'gray_to_darkblue',
    [(0.00, '#aaaaaa'),
     (0.40, "#106cc7"),
     (1.00, "#00172E")],
)
water_norm = mcolors.Normalize(vmin=0, vmax=global_max_depth)


def build_legend_handles(shelters_ids, exits_ids):
    handles = []
    handles.append(mpatches.Patch(color='#cccccc', label='Network node'))
    handles.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='#1565c0',
                          markersize=7, label='Flow → shelter'))
    if exits_ids:
        handles.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff6f00',
                              markersize=7, label='Flow → exit'))
    handles.append(Line2D([0], [0], marker='^', color='w', markerfacecolor='darkorange',
                          markersize=12, label='Shelter'))
    if exits_ids:
        handles.append(Line2D([0], [0], marker='*', color='w', markerfacecolor='gold',
                              markersize=11, label='Exit'))
    handles.append(Line2D([0], [0], color='#6699cc', linewidth=1.5, label='Open edge'))
    handles.append(Line2D([0], [0], color='red',     linewidth=1.5, label='Closed edge'))
    return handles


legend_handles = build_legend_handles(shelters_ids, exits_ids)

In [ ]:
# Render helpers & frame function
def _draw_edges(ax, open_segs, open_depths, closed_segs):
    if open_segs:
        ax.add_collection(LineCollection(
            [[(x0, y0), (x1, y1)] for x0, y0, x1, y1 in open_segs],
            colors=[water_cmap(water_norm(d)) for d in open_depths],
            linewidths=VIZ.EDGE_LW, alpha=VIZ.EDGE_ALPHA, zorder=1,
        ))
    if closed_segs:
        ax.add_collection(LineCollection(
            [[(x0, y0), (x1, y1)] for x0, y0, x1, y1 in closed_segs],
            colors='red', linewidths=VIZ.CLOSED_LW, alpha=VIZ.CLOSED_ALPHA, zorder=2,
        ))


def _draw_nodes(ax, ti):
    # ── layer 1: small grey base dot for every regular node ───────────────────
    if reg_mask.any():
        ax.scatter(
            xs[reg_mask], ys[reg_mask],
            c='#cccccc', s=VIZ.BASE_NODE_SIZE,
            zorder=3, edgecolors='none', linewidths=0,
        )

    # ── layer 2: moving agent dots interpolated along real edge geometry ──────
    # blue = shelter-bound,  orange = exit-bound  (same as original animation)
    if agent_shelter[ti]:
        ax_s, ay_s = zip(*agent_shelter[ti])
        ax.scatter(ax_s, ay_s, color="#00695c", s=VIZ.AGENT_SIZE,
                   alpha=0.85, zorder=4, edgecolors='none')
    if agent_exit[ti]:
        ax_e, ay_e = zip(*agent_exit[ti])
        ax.scatter(ax_e, ay_e, color='#ad1457', s=VIZ.AGENT_SIZE,
                   alpha=0.85, zorder=4, edgecolors='none')

    # ── layer 3: shelter / exit markers always on top ─────────────────────────
    if sh_mask.any():
        ax.scatter(
            xs[sh_mask], ys[sh_mask],
            marker='^', s=VIZ.SYMBOL_SIZE,
            color='darkorange', edgecolors='black', linewidths=0.2, zorder=5,
        )
    if ex_mask.any():
        ax.scatter(
            xs[ex_mask], ys[ex_mask],
            marker='*', s=VIZ.SYMBOL_SIZE,
            color='gold', edgecolors='black', linewidths=0.2, zorder=5,
        )


def _draw_colorbar(fig, cax):
    sm = cm.ScalarMappable(cmap=water_cmap, norm=water_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Flood depth (m)', color='black', fontsize=9)
    cbar.ax.yaxis.set_tick_params(color='black')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='black', fontsize=7)
    cbar.ax.set_facecolor('white')
    # axhline on colorbar axis uses data coordinates (depth values, not fractions)
    cbar.ax.axhline(cfg.IMPASSABLE_FLOOD_DEPTH, color='red', linewidth=1.5, linestyle='--')
    thresh_frac = water_norm(cfg.IMPASSABLE_FLOOD_DEPTH)
    cbar.ax.text(
        -0.9, thresh_frac,
        f' {cfg.IMPASSABLE_FLOOD_DEPTH} m\n(closed)',
        transform=cbar.ax.transAxes,
        color='red', fontsize=8, va='center', ha="center",
    )


def render_frame(ti: int) -> Image.Image:
    open_segs, open_depths, closed_segs, avg_depth, max_depth = all_edge_stats[ti]

    fig = plt.figure(figsize=(VIZ.FIG_W, VIZ.FIG_H), dpi=VIZ.DPI, facecolor='white')
    gs  = fig.add_gridspec(1, 2, width_ratios=[30, 1], wspace=0.03)
    ax  = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])

    ax.set_facecolor('white')
    ax.set_aspect('equal')
    ax.axis('off')

    _draw_edges(ax, open_segs, open_depths, closed_segs)
    _draw_nodes(ax, ti)

    ax.legend(
        handles=legend_handles, loc='lower left', ncol=2,
        framealpha=0.9, fontsize=9,
        facecolor='white', labelcolor='black',
    )

    ts_label = str(time_stamps[ti]) if ti < len(time_stamps) else '—'
    ax.set_title(
        f'Step {ti:>3d}/{len(time_names)-1}   '
        f'│   {ts_label}   '
        f'│   avg flood: {avg_depth:.3f} m   '
        f'│   max: {max_depth:.3f} m',
        color='black', fontsize=12
    )

    _draw_colorbar(fig, cax)
    ax.autoscale_view()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()


# preview frame 0
render_frame(0)

In [ ]:
from tqdm.auto import tqdm

frames = [render_frame(ti) for ti in tqdm(range(len(time_names)), desc='Rendering')]

frames[0].save(
    folder_path / OUTPUT_GIF,
    save_all=True, append_images=frames[1:],
    duration=VIZ.FRAME_DURATION, loop=0, optimize=False,
)

print(f"GIF saved → {folder_path / OUTPUT_GIF}  ({(folder_path / OUTPUT_GIF).stat().st_size / 1e6:.1f} MB)")

### Evacuation Cumulation

In [ ]:
import matplotlib.ticker as mticker

def plot_evacuation_timeseries(
    flow_dict,
    nodes,
    edges,
    time_stamps,
    time_names,
    shelter_flow_people,
    result_folder,
    impassable_depth=cfg.IMPASSABLE_FLOOD_DEPTH,
    figsize=(16, 11),
):
    _thai_fonts = [f.name for f in fm.fontManager.ttflist if "tahoma" in f.name.lower()]
    if _thai_fonts:
        plt.rcParams["font.family"] = _thai_fonts[0]
    else:
        for _fn in ["Browallia New", "Angsana New", "Cordia New", "Leelawadee UI"]:
            if any(_fn.lower() in f.name.lower() for f in fm.fontManager.ttflist):
                plt.rcParams["font.family"] = _fn
                break

    n_steps = len(time_names)
    flood_cols = [c for c in edges.columns if c.startswith("flood_depth_t")]
    # ensure sorted t0, t1, ... t{n}
    flood_cols = sorted(flood_cols, key=lambda c: int(c.split("_t")[-1]))

    shelter_nodes = (
        nodes[nodes["is_shelter"] == True]
        .drop_duplicates("osmid")
        .set_index("osmid")
    )
    shelter_osmids = list(shelter_nodes.index.astype(int))
    shelter_names  = {
        int(osmid): str(row.get("shelter_names", f"Shelter {osmid}"))
        for osmid, row in shelter_nodes.iterrows()
    }

    # ── Pre-compute per-timestep metrics ─────────────────────────────────────

    # 1. Incremental flow arriving at any collector per layer
    layer_flow = np.zeros(n_steps)
    shelter_layer_flow = {s: np.zeros(n_steps) for s in shelter_osmids}
    exit_layer_flow    = np.zeros(n_steps)

    exit_osmids = (
        set(nodes.loc[nodes["is_exit"] == True, "osmid"].astype(int))
        if "is_exit" in nodes.columns else set()
    )

    for u, nbrs in flow_dict.items():
        if not isinstance(u, tuple):
            continue
        osmid_u, layer_u = u
        if layer_u >= n_steps:
            continue
        for v, f in nbrs.items():
            if f <= 0 or not isinstance(v, str):
                continue
            layer_flow[layer_u] += f
            if v.startswith("shelter_"):
                sid = int(v.split("_")[1])
                if sid in shelter_layer_flow:
                    shelter_layer_flow[sid][layer_u] += f
            elif v.startswith("exit_"):
                exit_layer_flow[layer_u] += f

    shelter_cumulative = {s: np.cumsum(shelter_layer_flow[s]) for s in shelter_osmids}
    exit_cumulative    = np.cumsum(exit_layer_flow)
    total_cumulative   = np.cumsum(layer_flow)

    total_population = float(nodes["pop_total"].fillna(0).sum())
    stranded_over_time = total_population - total_cumulative

    # 2. Flood metrics per timestep from edges
    n_flood_cols = min(len(flood_cols), n_steps)
    avg_flood_depth   = np.zeros(n_flood_cols)
    flooded_edge_count = np.zeros(n_flood_cols)
    total_edges = len(edges)

    for i, col in enumerate(flood_cols[:n_flood_cols]):
        depths = edges[col].fillna(0).values
        avg_flood_depth[i]    = depths.mean()
        flooded_edge_count[i] = (depths >= impassable_depth).sum()

    flooded_pct = flooded_edge_count / total_edges * 100

    # 3. Shelter utilization % per timestep
    shelter_caps = {
        int(osmid): float(row["shelter_capacity"])
        for osmid, row in shelter_nodes.iterrows()
        if pd.notna(row.get("shelter_capacity"))
    }
    # util[shelter][timestep] = cumulative received / capacity
    util_matrix = np.zeros((len(shelter_osmids), n_steps))
    for si, osmid in enumerate(shelter_osmids):
        cap = shelter_caps.get(osmid, None)
        if cap and cap > 0:
            util_matrix[si] = np.clip(shelter_cumulative[osmid] / cap * 100, 0, 110)
        else:
            util_matrix[si] = np.zeros(n_steps)

    # ── Colors ───────────────────────────────────────────────────────────────
    shelter_colors = plt.cm.tab10(np.linspace(0, 1, len(shelter_osmids)))
    x = np.arange(n_steps)

    # ── Figure ───────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=figsize, facecolor="white")
    fig.suptitle(
        "Evacuation Simulation — Time-Series Analysis",
        fontsize=13, fontweight="bold", y=0.98,
    )

    # ════════════════════════════════════════════════════════════════════════
    # Panel A — Cumulative evacuated (stacked) + stranded
    # ════════════════════════════════════════════════════════════════════════
    ax = axes[0, 0]
    ax.set_facecolor("white")

    # stacked area: shelters bottom-up, then exits on top
    bottom = np.zeros(n_steps)
    for si, osmid in enumerate(shelter_osmids):
        ax.fill_between(
            x, bottom, bottom + shelter_cumulative[osmid],
            alpha=0.75, color=shelter_colors[si],
            label=shelter_names[osmid],
        )
        bottom += shelter_cumulative[osmid]
    ax.fill_between(
        x, bottom, bottom + exit_cumulative,
        alpha=0.75, color="slategrey", label="Exits",
    )

    # stranded on right axis
    ax2 = ax.twinx()
    ax2.plot(x, stranded_over_time, color="crimson", linewidth=1.8,
             linestyle="--", label="Stranded remaining")
    ax2.set_ylabel("Stranded (people)", fontsize=9, color="crimson")
    ax2.tick_params(axis="y", labelcolor="crimson", labelsize=7)
    ax2.set_ylim(bottom=0)

    ax.set_title("A — Cumulative Evacuated & Stranded", fontsize=10)
    ax.set_xlabel("Timestep", fontsize=9)
    ax.set_ylabel("People evacuated", fontsize=9)
    ax.set_xlim(0, n_steps - 1)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

    # combined legend
    handles_a, labels_a = ax.get_legend_handles_labels()
    handles_b, labels_b = ax2.get_legend_handles_labels()
    ax.legend(handles_a + handles_b, labels_a + labels_b,
              fontsize=7.5, loc="upper left", framealpha=0.85)

    # ════════════════════════════════════════════════════════════════════════
    # Panel B — Flood depth + flooded edge % + impassable threshold
    # ════════════════════════════════════════════════════════════════════════
    ax = axes[0, 1]
    ax.set_facecolor("white")
    xf = np.arange(n_flood_cols)

    color_depth = "#2166ac"
    color_pct   = "#d73027"

    ax.fill_between(xf, avg_flood_depth, alpha=0.35, color=color_depth)
    ax.plot(xf, avg_flood_depth, color=color_depth, linewidth=1.8,
            label="Avg flood depth (m)")
    ax.axhline(impassable_depth, color=color_depth, linewidth=1,
               linestyle=":", alpha=0.8, label=f"Impassable threshold ({impassable_depth} m)")
    ax.set_ylabel("Avg flood depth (m)", fontsize=9, color=color_depth)
    ax.tick_params(axis="y", labelcolor=color_depth, labelsize=7)

    ax3 = ax.twinx()
    ax3.plot(xf, flooded_pct, color=color_pct, linewidth=1.8,
             linestyle="-.", label="Flooded edges (%)")
    ax3.fill_between(xf, flooded_pct, alpha=0.15, color=color_pct)
    ax3.set_ylabel("Flooded edges (%)", fontsize=9, color=color_pct)
    ax3.tick_params(axis="y", labelcolor=color_pct, labelsize=7)
    ax3.set_ylim(0, 100)

    ax.set_title("B — Flood Depth & Network Closure Over Time", fontsize=10)
    ax.set_xlabel("Timestep", fontsize=9)
    ax.set_xlim(0, n_flood_cols - 1)
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

    handles_c  = [
        Line2D([0], [0], color=color_depth, linewidth=1.8, label="Avg flood depth (m)"),
        Line2D([0], [0], color=color_depth, linewidth=1, linestyle=":",
               label=f"Impassable ({impassable_depth} m)"),
        Line2D([0], [0], color=color_pct, linewidth=1.8, linestyle="-.",
               label="Flooded edges (%)"),
    ]
    ax.legend(handles=handles_c, fontsize=7.5, loc="upper left", framealpha=0.85)

    # ════════════════════════════════════════════════════════════════════════
    # Panel C — Per-shelter cumulative inflow
    # ════════════════════════════════════════════════════════════════════════
    ax = axes[1, 0]
    ax.set_facecolor("white")

    for si, osmid in enumerate(shelter_osmids):
        cap = shelter_caps.get(osmid)
        ax.plot(
            x, shelter_cumulative[osmid],
            color=shelter_colors[si], linewidth=1.8,
            label=shelter_names[osmid],
        )
        if cap and cap > 0:
            ax.axhline(
                cap, color=shelter_colors[si],
                linewidth=0.8, linestyle="--", alpha=0.6,
            )

    ax.set_title("C — Per-Shelter Cumulative Inflow", fontsize=10)
    ax.set_xlabel("Timestep", fontsize=9)
    ax.set_ylabel("People received", fontsize=9)
    ax.set_xlim(0, n_steps - 1)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

    # capacity legend note
    cap_patch = Line2D([0], [0], linewidth=0.8, linestyle="--",
                       color="grey", label="── capacity limit")
    handles_d, labels_d = ax.get_legend_handles_labels()
    ax.legend(handles_d + [cap_patch], labels_d + ["capacity limit"],
              fontsize=7.5, loc="upper left", framealpha=0.85)

    # ════════════════════════════════════════════════════════════════════════
    # Panel D — Shelter utilization heatmap (shelter × timestep)
    # ════════════════════════════════════════════════════════════════════════
    ax = axes[1, 1]

    im = ax.imshow(
        util_matrix,
        aspect="auto", cmap="RdYlGn_r",
        vmin=0, vmax=100,
        interpolation="nearest",
    )

    # over-capacity contour line at 100%
    ax.contour(
        util_matrix, levels=[100],
        colors="red", linewidths=1.2, linestyles="--",
    )

    ax.set_title("D — Shelter Utilization % Over Time", fontsize=10)
    ax.set_xlabel("Timestep", fontsize=9)
    ax.set_ylabel("Shelter", fontsize=9)
    ax.set_yticks(range(len(shelter_osmids)))
    ax.set_yticklabels(
        [shelter_names[s] for s in shelter_osmids],
        fontsize=7,
    )
    ax.tick_params(axis="x", labelsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=8))

    cbar_d = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar_d.set_label("Utilization %", fontsize=9)
    cbar_d.ax.tick_params(labelsize=7)

    red_line = Line2D([0], [0], color="red", linewidth=1.2,
                      linestyle="--", label="100% capacity")
    ax.legend(handles=[red_line], fontsize=7.5,
              loc="lower right", framealpha=0.85)

    # ── Save ─────────────────────────────────────────────────────────────────
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    out_path = result_folder / "evacuation_timeseries.png"
    fig.savefig(out_path, dpi=300, facecolor="white")
    print(f"Saved: {out_path}")
    plt.show()
    return fig


# ── Run ───────────────────────────────────────────────────────────────────────
fig = plot_evacuation_timeseries(
    flow_dict=flow_result["flow_dict"],
    nodes=nodes_df,
    edges=edges_df,
    time_stamps=time_stamps,
    time_names=time_names,
    shelter_flow_people=flow_result["shelter_flow_people"],
    result_folder=folder_path,
)